# Agent B Sanity Check

This notebook tests Agent B behavior with mocked Agent A payloads.

It does not run the full extraction pipeline. Instead, it feeds sample `Agent A` outputs directly into a small `Agent B` harness so we can verify:
- pass handling
- fail and retry behavior
- human review routing when `retry_count >= 2`
- schema preservation for `ci` and `iqr`
- empty-verdict fallback behavior before escalation
- explicit assertions for expected outcomes


In [9]:
import copy
from pprint import pprint


In [10]:
def parse_pass_fail(text: str):
    t = (text or "").strip()
    first = t.splitlines()[0].strip().upper() if t else ""
    if "PASS" in first:
        return "PASS", "Accepted by verifier"
    if "FAIL" in first:
        reason = t.split(":", 1)[1].strip() if ":" in t else "Verifier rejected extraction"
        return "FAIL", reason
    if "PASS" in t.upper():
        return "PASS", "Accepted by verifier"
    if "FAIL" in t.upper():
        return "FAIL", "Verifier rejected extraction"
    return "FAIL", f"Verifier unclear: {t[:120]}"


def normalize_extraction(extraction: dict):
    if not isinstance(extraction, dict):
        return None
    metric = str(extraction.get("metric_type", "")).strip().upper()
    metric = "R0" if metric == "R0" else ("Rt" if metric == "RT" else extraction.get("metric_type"))
    try:
        value = float(extraction.get("value"))
    except Exception:
        value = extraction.get("value")
    return {
        "disease": extraction.get("disease"),
        "location": extraction.get("location"),
        "metric_type": metric,
        "value": value,
        "time_frame": extraction.get("time_frame"),
        "context_label": extraction.get("context_label"),
        "ci": extraction.get("ci"),
        "iqr": extraction.get("iqr"),
        "evidence_snippet": extraction.get("evidence_snippet"),
    }


In [11]:
def agent_b_step(payload, final_database, human_review_database, retry_buffer, verdict_fn, fallback_verdict_fn=None):
    """Small single-step harness that mirrors the patched notebook Agent B logic."""
    metadata = payload.get("metadata", {})
    agent_a_error = payload.get("agent_a_error")
    no_evidence = payload.get("no_evidence", False)

    extractions = payload.get("extractions")
    if extractions is None:
        single = payload.get("extraction")
        if isinstance(single, dict):
            extractions = [single]
        else:
            extractions = []

    if agent_a_error:
        status, reason = "FAIL", agent_a_error
        raw_verdict = f"FAIL: {reason}"
    else:
        raw_verdict = (verdict_fn(payload) or "").strip()
        if not raw_verdict:
            fallback_verdict_fn = fallback_verdict_fn or (lambda _payload: "")
            raw_verdict = (fallback_verdict_fn(payload) or "").strip()
            if not raw_verdict:
                status, reason = "FAIL", "Verifier empty output"
                raw_verdict = "FAIL: Verifier empty output"
            else:
                status, reason = parse_pass_fail(raw_verdict)
        else:
            status, reason = parse_pass_fail(raw_verdict)

    if status == "PASS":
        if len(extractions) > 0:
            for ext in extractions:
                fixed = normalize_extraction(ext)
                final_record = {
                    "article_id": metadata.get("pmid"),
                    "year": metadata.get("year"),
                    "disease": fixed.get("disease"),
                    "location": fixed.get("location"),
                    "metric": fixed.get("metric_type"),
                    "value": fixed.get("value"),
                    "time_frame": fixed.get("time_frame"),
                    "context_label": fixed.get("context_label"),
                    "ci": fixed.get("ci"),
                    "iqr": fixed.get("iqr"),
                    "evidence_snippet": fixed.get("evidence_snippet"),
                    "agent_a_extraction": ext,
                    "agent_b_verdict": raw_verdict,
                }
                final_database.append(final_record)
            action = f"PASS ({len(extractions)} extraction(s) accepted)"
        else:
            final_database.append({
                "article_id": metadata.get("pmid"),
                "year": metadata.get("year"),
                "disease": None,
                "location": None,
                "metric": None,
                "value": None,
                "time_frame": None,
                "context_label": None,
                "ci": None,
                "iqr": None,
                "evidence_snippet": None,
                "agent_a_extraction": None,
                "agent_b_verdict": raw_verdict,
                "no_evidence": True,
            })
            action = "PASS-NO-EVIDENCE"
    else:
        retry_count = int(metadata.get("retry_count", 0))
        if retry_count >= 2:
            fail_item = dict(metadata)
            fail_item["failure_reason"] = reason
            fail_item["agent_a_extractions"] = extractions
            fail_item["agent_b_verdict"] = raw_verdict
            fail_item["no_evidence"] = no_evidence
            human_review_database.append(fail_item)
            action = "HUMAN_REVIEW"
        else:
            retried = dict(metadata)
            retried["retry_count"] = retry_count + 1
            retried["audit_feedback"] = f"Agent B flagged this abstract-level extraction: {reason}"
            retry_buffer.append(retried)
            action = f"RETRY -> retry_count {retried['retry_count']}"

    return {
        "pmid": metadata.get("pmid"),
        "status": status,
        "reason": reason,
        "raw_verdict": raw_verdict,
        "action": action,
        "retry_count": metadata.get("retry_count", 0),
    }


In [12]:
sample_payloads = {
    "pass_with_iqr": {
        "metadata": {
            "pmid": "PASS001",
            "year": "2014",
            "text": "The median R value for 1918 was 1.80 (interquartile range [IQR]: 1.47-2.27).",
            "retry_count": 0,
        },
        "extractions": [{
            "disease": "Influenza",
            "location": "Global",
            "metric_type": "R0",
            "value": "1.80",
            "time_frame": "1918 pandemic",
            "context_label": "",
            "ci": "",
            "iqr": "interquartile range [IQR]: 1.47-2.27",
            "evidence_snippet": "The median R value for 1918 was 1.80 (interquartile range [IQR]: 1.47-2.27).",
        }],
    },
    "fail_retry": {
        "metadata": {
            "pmid": "FAIL001",
            "year": "2020",
            "text": "R0 was 3.1 in Italy during March 2020.",
            "retry_count": 0,
        },
        "extractions": [{
            "disease": "COVID-19",
            "location": "Italy",
            "metric_type": "R0",
            "value": "3.1",
            "time_frame": "",
            "context_label": "",
            "ci": "",
            "iqr": "",
            "evidence_snippet": "R0 was 3.1 in Italy.",
        }],
    },
    "pass_no_evidence": {
        "metadata": {
            "pmid": "PASS002",
            "year": "2021",
            "text": "The paper discusses surveillance methods only.",
            "retry_count": 0,
        },
        "extraction": None,
        "no_evidence": True,
    },
    "agent_a_error": {
        "metadata": {
            "pmid": "ERR001",
            "year": "2022",
            "text": "The abstract contains malformed output from Agent A.",
            "retry_count": 2,
        },
        "extraction": None,
        "agent_a_error": "Invalid or empty extraction JSON from Agent A",
    },
    "empty_then_pass": {
        "metadata": {
            "pmid": "EMPTYPASS001",
            "year": "2018",
            "text": "R0 was 2.2 in a hospital outbreak in April 2018.",
            "retry_count": 0,
        },
        "extractions": [{
            "disease": "SARS",
            "location": "hospital",
            "metric_type": "R0",
            "value": "2.2",
            "time_frame": "April 2018",
            "context_label": "",
            "ci": "",
            "iqr": "",
            "evidence_snippet": "R0 was 2.2 in a hospital outbreak in April 2018.",
        }],
    },
    "empty_twice": {
        "metadata": {
            "pmid": "EMPTYFAIL001",
            "year": "2019",
            "text": "Rt was 0.9 after interventions.",
            "retry_count": 0,
        },
        "extractions": [{
            "disease": "Ebola",
            "location": "Liberia",
            "metric_type": "Rt",
            "value": "0.9",
            "time_frame": "after interventions",
            "context_label": "",
            "ci": "",
            "iqr": "",
            "evidence_snippet": "Rt was 0.9 after interventions.",
        }],
    },
}

def scripted_verdict(payload):
    pmid = payload.get("metadata", {}).get("pmid")
    verdict_map = {
        "PASS001": "PASS",
        "FAIL001": "FAIL: time_frame missing March 2020",
        "PASS002": "PASS",
        "EMPTYPASS001": "",
        "EMPTYFAIL001": "",
    }
    return verdict_map.get(pmid, "FAIL: no scripted verdict")

def scripted_fallback_verdict(payload):
    pmid = payload.get("metadata", {}).get("pmid")
    verdict_map = {
        "EMPTYPASS001": "PASS",
        "EMPTYFAIL001": "",
    }
    return verdict_map.get(pmid, "")


In [13]:
results_db = []
human_review_db = []
retry_buffer = []

for name, payload in sample_payloads.items():
    summary = agent_b_step(
        copy.deepcopy(payload),
        results_db,
        human_review_db,
        retry_buffer,
        scripted_verdict,
        scripted_fallback_verdict,
    )
    print(f"\nCASE: {name}")
    pprint(summary)

print("\nvalidated records:", len(results_db))
print("retry queue:", len(retry_buffer))
print("human review:", len(human_review_db))



CASE: pass_with_iqr
{'action': 'PASS (1 extraction(s) accepted)',
 'pmid': 'PASS001',
 'raw_verdict': 'PASS',
 'reason': 'Accepted by verifier',
 'retry_count': 0,
 'status': 'PASS'}

CASE: fail_retry
{'action': 'RETRY -> retry_count 1',
 'pmid': 'FAIL001',
 'raw_verdict': 'FAIL: time_frame missing March 2020',
 'reason': 'time_frame missing March 2020',
 'retry_count': 0,
 'status': 'FAIL'}

CASE: pass_no_evidence
{'action': 'PASS-NO-EVIDENCE',
 'pmid': 'PASS002',
 'raw_verdict': 'PASS',
 'reason': 'Accepted by verifier',
 'retry_count': 0,
 'status': 'PASS'}

CASE: agent_a_error
{'action': 'HUMAN_REVIEW',
 'pmid': 'ERR001',
 'raw_verdict': 'FAIL: Invalid or empty extraction JSON from Agent A',
 'reason': 'Invalid or empty extraction JSON from Agent A',
 'retry_count': 2,
 'status': 'FAIL'}

CASE: empty_then_pass
{'action': 'PASS (1 extraction(s) accepted)',
 'pmid': 'EMPTYPASS001',
 'raw_verdict': 'PASS',
 'reason': 'Accepted by verifier',
 'retry_count': 0,
 'status': 'PASS'}

CASE

In [14]:
# This cell shows the exact escalation behavior across repeated Agent B failures.
repeated_fail = copy.deepcopy(sample_payloads["fail_retry"])
results_db = []
human_review_db = []
retry_buffer = []

for attempt in range(1, 5):
    summary = agent_b_step(
        repeated_fail,
        results_db,
        human_review_db,
        retry_buffer,
        scripted_verdict,
        scripted_fallback_verdict,
    )
    print(f"attempt {attempt} -> {summary['action']} | raw_verdict={summary['raw_verdict']}")
    if retry_buffer:
        repeated_fail = {
            "metadata": retry_buffer.pop(0),
            "extractions": copy.deepcopy(sample_payloads["fail_retry"]["extractions"]),
        }
    if human_review_db:
        break

print("\nfinal retry_count in human review item:")
pprint(human_review_db[0]["retry_count"] if human_review_db else None)


attempt 1 -> RETRY -> retry_count 1 | raw_verdict=FAIL: time_frame missing March 2020
attempt 2 -> RETRY -> retry_count 2 | raw_verdict=FAIL: time_frame missing March 2020
attempt 3 -> HUMAN_REVIEW | raw_verdict=FAIL: time_frame missing March 2020

final retry_count in human review item:
2


In [15]:
# Assertions so the sanity check fails loudly if Agent B behavior changes.
results_db = []
human_review_db = []
retry_buffer = []

case_results = {}
for name, payload in sample_payloads.items():
    case_results[name] = agent_b_step(
        copy.deepcopy(payload),
        results_db,
        human_review_db,
        retry_buffer,
        scripted_verdict,
        scripted_fallback_verdict,
    )

assert case_results["pass_with_iqr"]["action"].startswith("PASS")
assert case_results["pass_no_evidence"]["action"] == "PASS-NO-EVIDENCE"
assert case_results["fail_retry"]["action"] == "RETRY -> retry_count 1"
assert case_results["agent_a_error"]["action"] == "HUMAN_REVIEW"
assert case_results["empty_then_pass"]["action"].startswith("PASS")
assert case_results["empty_twice"]["action"] == "RETRY -> retry_count 1"

accepted_iqr = [r for r in results_db if r.get("article_id") == "PASS001"][0]
assert accepted_iqr["iqr"] == "interquartile range [IQR]: 1.47-2.27"

print("All Agent B sanity assertions passed.")


All Agent B sanity assertions passed.


In [16]:
# Optional: inspect accepted records.
pprint(results_db)


[{'agent_a_extraction': {'ci': '',
                         'context_label': '',
                         'disease': 'Influenza',
                         'evidence_snippet': 'The median R value for 1918 was '
                                             '1.80 (interquartile range [IQR]: '
                                             '1.47-2.27).',
                         'iqr': 'interquartile range [IQR]: 1.47-2.27',
                         'location': 'Global',
                         'metric_type': 'R0',
                         'time_frame': '1918 pandemic',
                         'value': '1.80'},
  'agent_b_verdict': 'PASS',
  'article_id': 'PASS001',
  'ci': '',
  'context_label': '',
  'disease': 'Influenza',
  'evidence_snippet': 'The median R value for 1918 was 1.80 (interquartile '
                      'range [IQR]: 1.47-2.27).',
  'iqr': 'interquartile range [IQR]: 1.47-2.27',
  'location': 'Global',
  'metric': 'R0',
  'time_frame': '1918 pandemic',
  'value': 1.8,
 